In [79]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer

### dummy-encode movie genres:

In [80]:
movies = pd.read_csv('ml-latest/movies.csv')
movies['genres'] = movies['genres'].str.split('|')
movies['genres'].head()

0    [Adventure, Animation, Children, Comedy, Fantasy]
1                       [Adventure, Children, Fantasy]
2                                    [Comedy, Romance]
3                             [Comedy, Drama, Romance]
4                                             [Comedy]
Name: genres, dtype: object

In [81]:
mlb = MultiLabelBinarizer()

movies_encoded = movies.join(pd.DataFrame(mlb.fit_transform(movies['genres']), columns=mlb.classes_))
movies_encoded.head()

,movieId,title,genres,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",0,0,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",0,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",0,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),[Comedy],0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [82]:
movies_encoded.to_csv('ml-latest/movies_encoded.csv')

### convert tags into vectors, TF-IDF (test token embedding llm)

In [83]:
tags = pd.read_csv('ml-latest/tags.csv')
tags.head()

,userId,movieId,tag,timestamp
0,10,260,good vs evil,1430666558
1,10,260,Harrison Ford,1430666505
2,10,260,sci-fi,1430666538
3,14,1221,Al Pacino,1311600756
4,14,1221,mafia,1311600746


combine all tags for movies into long strings in a pd series with movieId as index (for TF-IDF)

In [84]:
tags_tfidf = tags.dropna(axis=0).groupby('movieId')['tag'].apply(lambda x: ' '.join(x).lower())

create TfidfVectorizer class, fit_transform on our strings, create DF with vectorized strings

In [85]:
TFIDF_vectorizer = TfidfVectorizer(max_features=2500)

tag_vectorized_matrix = TFIDF_vectorizer.fit_transform(tags_tfidf).toarray()

tags_vectorized_tfidf = pd.DataFrame(tag_vectorized_matrix, columns=TFIDF_vectorizer.get_feature_names_out())

In [86]:
tags_vectorized_tfidf.info()

<class 'pandas.DataFrame'>
RangeIndex: 53452 entries, 0 to 53451
Columns: 2500 entries, 007 to zooey
dtypes: float64(2500)
memory usage: 1019.5 MB
